# 04 — Modeling (CRISP-DM)

O objetivo de negócio não é "prever vendas", é **decidir o desconto** que
maximiza o lucro. A modelagem separa duas peças:

1. **Modelo de demanda** — aprende `quantidade_vendida_dia` a partir das
   variáveis de decisão (preço/desconto) e do contexto (loja, produto, clima,
   calendário).
2. **Otimizador de preço** — usa o modelo de demanda para escolher, por linha,
   o desconto (de uma grade) que maximiza o lucro esperado
   `(preço_final − custo) × demanda_prevista`, respeitando o piso `preco_minimo`.

Todo o pipeline vive em código testável (`src/modeling/train_pipeline.py`);
este notebook apenas narra e inspeciona o processo.


## 1) Dados e contrato de features (leakage-safe)

Consumimos a base preparada da etapa anterior. Antes de qualquer coisa, o
contrato de features é validado *fail-closed*: se alguma coluna vetada por
vazamento entrar no contrato, o treino aborta.

In [ ]:
from src.modeling import train_pipeline as tp

df = tp.carregar_base_preparada()
tp.validar_contrato_features(df)  # fail-closed contra vazamento de dados
print(f"Base: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
print("Colunas vetadas por vazamento:", tp.colunas_com_vazamento())

## 2) Protocolo temporal (passado → futuro)

A avaliação é sempre passado → futuro, sem embaralhar:

- **treino** (≤ 2025-06-30): ajuste dos candidatos;
- **validação** (jul–set/2025): seleção do modelo por lucro esperado;
- **teste** (Q4/2025, o pico festivo): relatório final, nunca usado na seleção.

In [ ]:
splits = tp.separar_splits(df)
for nome, parte in splits.items():
    inicio, fim = parte["data"].min().date(), parte["data"].max().date()
    print(f"{nome:>9}: {len(parte):>6,} linhas | {inicio} → {fim}")

## 3) Grade de desconto no suporte identificado

A grade é limitada a 0–12%: é o teto do desconto de balcão **sem promoção**
(`U(0,03; 0,12)`), a única região em que o contrafactual "desconto sem promoção"
está identificado nos dados. Acima disso, os descontos observados vêm sempre
atrelados a um lift promocional e os aprendizes de árvore extrapolam o lucro
para cima — uma forma sutil de "vazamento" contrafactual.

In [ ]:
print("Grade avaliada (desconto):", tp.GRADE_DESCONTO.tolist())

## 4) Treino, seleção e publicação

`main()` ajusta os candidatos no treino, seleciona o campeão por **lucro
esperado na validação**, re-ajusta em treino+validação (mais passado, mesmo
teste intocado) e publica os artefatos.

In [ ]:
tp.main()

## 5) Comparativo dos candidatos (validação)

Ordenado pelo critério de seleção — o lucro esperado da política ótima.

In [ ]:
import json

import pandas as pd

from src.config.paths import MODELS_DIR

snapshot = json.loads((MODELS_DIR / "metrics_snapshot.json").read_text(encoding="utf-8"))
colunas = ["rmse", "mae", "r2", "lucro_politica", "uplift_pct", "desconto_medio"]
comparativo = pd.DataFrame(
    [{"modelo": c["modelo"], **{k: c["validacao"][k] for k in colunas}} for c in snapshot["candidatos"]]
).sort_values("lucro_politica", ascending=False)
comparativo.round(4)

In [ ]:
print(f"Campeão: {snapshot['modelo']}")
print(f"Critério: maior lucro esperado na validação ({snapshot['modelo']})")

## Conclusão

O campeão é o candidato com maior lucro esperado na validação temporal, com o
teste (Q4) intocado. Os artefatos — `best_model.pkl`, `metrics_snapshot.json`,
`model_card.json`, `business_curve.csv` e `test_predictions.parquet` — ficam em
`models/`, governados pelo DVC, e alimentam a avaliação do notebook 05.